In [ ]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from qso import sdss

In [ ]:
DATA_DIR = "data/"
SDSS_DR14 = DATA_DIR + "sdss_dr14/"
SDSS_CATALOG = DATA_DIR + "specObj-dr14.fits"
SDSS_QSO_CATALOG = DATA_DIR + "DR14Q_v4_4.fits"
FIGS_DIR = "figs/"

In [ ]:
# wavelengths from `lamost_analysis.ipynb`
WAVEMIN, WAVEMAX = 3839.7244, 8914.597

# SDSS DR14 Optical Spectra Catalog

According to [data model](https://data.sdss.org/datamodel/files/SPECTRO_REDUX/specObj.html)
contains the list of all spectra for a given data release,
with associated parameters from the 2D and 1D pipelines for each.

In [ ]:
with fits.open(SDSS_CATALOG) as catalog_hdul:
    catalog_hdul.info()
    catalog = catalog_hdul[1].data

n_spec = catalog.shape[0]

In [ ]:
catalog["plate"].dtype, catalog["mjd"].dtype, catalog["fiberid"].dtype

In [ ]:
# convert dtypes to machine native
arrays = [catalog["plate"].astype("i4"), catalog["mjd"].astype("i4"), catalog["fiberid"].astype("i4")]
multiindex = pd.MultiIndex.from_arrays(arrays, names=["plate", "mjd", "fiberid"])

data = {
    "platequality": pd.Categorical(catalog["platequality"]),
    "targettype": pd.Categorical(catalog["targettype"]),
    "wavemax": catalog["wavemax"].astype("f4"),
    "wavemin": catalog["wavemin"].astype("f4"),
    "zwarning": catalog["zwarning"].astype("i4")
}

catalog_df = pd.DataFrame(data, index=multiindex).sort_index()
catalog_df.info()
catalog_df.head()

In [ ]:
catalog_df.loc[catalog_df["wavemax"] == 0.0, "wavemax"] = np.nan
catalog_df.loc[catalog_df["wavemin"] == 0.0, "wavemin"] = np.nan

In [ ]:
zwarning = catalog_df["zwarning"].values

# https://www.sdss.org/dr13/algorithms/bitmasks/#ZWARNING
unplugged_idx = np.bitwise_and(zwarning, 2 ** 7).astype("bool")
nodata_idx = np.bitwise_and(zwarning, 2 ** 9).astype("bool")

catalog_df["unplugged"] = False
catalog_df.loc[unplugged_idx, "unplugged"] = True
catalog_df["nodata"] = False
catalog_df.loc[nodata_idx, "nodata"] = True

catalog_df.head()

# SDSS DR14 Quasar Catalog

Known problems of the catalog are listed in a Jupyter notebook
[Preparing SDSS DR14 VACs](https://github.com/noaodatalab/digestor/blob/master/docs/notebooks/Preparing_SDSS_DR14_VACs.ipynb)
where VACs value added catalogs.

In [ ]:
with fits.open(SDSS_QSO_CATALOG) as qso_hdul:
    qso_hdul.info()
    qso_catalog = qso_hdul[1].data

In [ ]:
n_primary = qso_catalog.shape[0]
n_additional = qso_catalog["n_spec"].sum()
n_primary, n_additional, n_primary + n_additional

In [ ]:
def get_duplicate_spec(duplicate):
    return duplicate[duplicate > 0]

plate_duplicate   = get_duplicate_spec(qso_catalog["plate_duplicate"])
mjd_duplicate     = get_duplicate_spec(qso_catalog["mjd_duplicate"])
fiberid_duplicate = get_duplicate_spec(qso_catalog["fiberid_duplicate"])

plate_duplicate.shape, mjd_duplicate.shape, fiberid_duplicate.shape

In [ ]:
n_true_additional = plate_duplicate.shape[0]
n_qso = n_primary + n_true_additional
n_true_additional, n_qso, (n_primary + n_additional) - n_qso

In [ ]:
qso_catalog["plate"].dtype, qso_catalog["mjd"].dtype, qso_catalog["fiberid"].dtype

In [ ]:
def fill_array(arr, primaries, duplicates, n_primary=n_primary):
    arr[:n_primary] = primaries
    arr[n_primary:] = duplicates

plate   = np.zeros(shape=n_qso, dtype='i2')
mjd     = np.zeros(shape=n_qso, dtype='i4')
fiberid = np.zeros(shape=n_qso, dtype='i2')

fill_array(plate, qso_catalog["plate"], plate_duplicate)
fill_array(mjd, qso_catalog["mjd"], mjd_duplicate)
fill_array(fiberid, qso_catalog["fiberid"], fiberid_duplicate)

# Combining

In [ ]:
qso_multiindex = pd.MultiIndex.from_arrays(
    [plate, mjd, fiberid],
    names=["plate", "mjd", "fiberid"]
)

qso_missing = qso_multiindex.difference(multiindex)
qso_multiindex = qso_multiindex.difference(qso_missing)
qso_multiindex.shape

In [ ]:
qso_missing.shape, qso_missing

In [ ]:
catalog_df["qso"] = False
catalog_df.loc[qso_multiindex, "qso"] = True
catalog_df.head()

In [ ]:
catalog_df["qso"].sum()

# Analysis

In [ ]:
def plot_spec(plate, mjd, fiberid):
    dr_path = sdss.get_dr_path(plate, mjd, fiberid)
    filepath = SDSS_DR14 + dr_path
    with fits.open(filepath) as hdul:
        data = hdul[1].data
        ax = plt.axes(xlabel="Wavelength", ylabel="Flux")
        ax.plot(np.power(10, data["loglam"]), data["flux"])

a_qso = catalog_df[catalog_df["qso"]].sample()
plot_spec(*a_qso.index[0])

In [ ]:
a_nodata_spec = catalog_df[catalog_df["nodata"]].sample()
plot_spec(*a_nodata_spec.index[0])

In [ ]:
np.all(catalog_df["wavemin"].isna() == catalog_df["wavemax"].isna())

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, sharey=True)
catalog_df["wavemin"].plot.hist(ax=ax1, bins=32, log=True)
catalog_df["wavemax"].plot.hist(ax=ax2, bins=32, log=True)
ax1.set_xlabel("Minimal wavelength (\AA{})")
ax2.set_xlabel("Maximal wavelength (\AA{})")

In [ ]:
fig, (ax1, ax2) = plt.subplots(nrows=2, sharex=True)
ax1.set_xlabel("Wavelengths (\AA{})")
ax2.set_xlabel("Wavelengths (\AA{})")
catalog_df["wavemin"].plot.hist(
    ax=ax1, histtype='step', bins=32, cumulative=True,
    label="Cumulative histogram of minimal wavelengths"
)
catalog_df["wavemax"].plot.hist(
    ax=ax2, histtype='step', bins=32, cumulative=-1,
    label="Cumulative histogram of maximal wavelengths"
)
axvline_style = {"linestyle": "dashed", "linewidth": 1, "color": "k"}
ax1.axvline(WAVEMIN, label=str(WAVEMIN) + " \AA{}", **axvline_style)
ax2.axvline(WAVEMAX, label=str(WAVEMAX) + " \AA{}", **axvline_style)
ax1.legend(loc="lower right")
ax2.legend(loc="lower left")
plt.savefig(FIGS_DIR + "waves_cumulative_hist.pdf")

In [ ]:
idx_selected = (catalog_df["wavemin"] <= WAVEMIN) & (WAVEMAX <= catalog_df["wavemax"])
n_selected = idx_selected.sum()
n_selected, n_spec - n_selected

In [ ]:
catalog_selected_df = catalog_df[idx_selected]
catalog_selected_df.shape, catalog_df.shape

In [ ]:
catalog_df["qso"].sum() - catalog_selected_df["qso"].sum()

In [ ]:
catalog_selected_df["qso"].sum()

# Dataset

In [ ]:
N_TRAIN = 1000000
N_VALID = 50000
SIZE = N_TRAIN + N_VALID

In [ ]:
sample_df = catalog_selected_df.sample(SIZE, random_state=9)
sample_df.iloc[:N_TRAIN]["qso"].sum(), sample_df.iloc[N_TRAIN:]["qso"].sum()